<a href="https://colab.research.google.com/github/vaslal-324/Circuit_Faith_Audit/blob/main/mech_int.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

"""
Mech Interp Lab — 4 runnable mini-projects on GPT-2 small using TransformerLens.

SETUP (Colab or local, CPU is fine for gpt2-small):
    pip install transformer_lens matplotlib torch

USAGE:
    python mech_interp_lab.py --project 1   # induction head detection
    python mech_interp_lab.py --project 2   # logit lens
    python mech_interp_lab.py --project 3   # head ablation
    python mech_interp_lab.py --project 4   # activation patching (IOI)
    python mech_interp_lab.py --project all  # run everything

Each project prints a short result summary and saves a .png plot to ./outputs/.
"""

import argparse
import os
import torch
import matplotlib.pyplot as plt
import numpy as np

os.makedirs("outputs", exist_ok=True)


def load_model():
    from transformer_lens import HookedTransformer
    print("Loading gpt2-small (first run downloads weights, ~500MB)...")
    model = HookedTransformer.from_pretrained("gpt2")
    model.eval()
    return model


# ---------------------------------------------------------------------------
# Project 1: Find an induction head
# ---------------------------------------------------------------------------
def project_1_induction_head(model, seq_len=25, seed=0):
    """
    Feed the model a repeated random token sequence [A B C ... A B C ...]
    and score every head by how strongly it attends to "the token that
    followed the current token, last time it appeared" — the induction
    pattern. This shows up as high attention at the (-seq_len + 1) offset.
    """
    torch.manual_seed(seed)
    random_tokens = torch.randint(1000, 10000, (1, seq_len))
    repeated_tokens = torch.cat([random_tokens, random_tokens], dim=1)

    _, cache = model.run_with_cache(repeated_tokens)

    n_layers, n_heads = model.cfg.n_layers, model.cfg.n_heads
    induction_scores = torch.zeros(n_layers, n_heads)

    for layer in range(n_layers):
        pattern = cache["pattern", layer]  # [batch, head, dest_pos, src_pos]
        pattern = pattern[0]  # drop batch dim
        seq_len_full = pattern.shape[-1]
        # score = avg attention paid from each dest position back to
        # (dest_pos - seq_len + 1), i.e. the token after the earlier occurrence
        offset = seq_len - 1
        scores = []
        for head in range(n_heads):
            diag_vals = []
            for dest in range(offset, seq_len_full):
                src = dest - offset
                diag_vals.append(pattern[head, dest, src].item())
            scores.append(np.mean(diag_vals))
        induction_scores[layer] = torch.tensor(scores)

    best_layer, best_head = np.unravel_index(
        induction_scores.argmax(), induction_scores.shape
    )
    best_score = induction_scores[best_layer, best_head].item()

    print(f"\n[Project 1] Best induction head: L{best_layer}H{best_head} "
          f"(score={best_score:.3f})")
    print("Top 5 heads by induction score:")
    flat = induction_scores.flatten()
    top5 = torch.topk(flat, 5)
    for score, idx in zip(top5.values, top5.indices):
        l, h = divmod(idx.item(), n_heads)
        print(f"  L{l}H{h}: {score.item():.3f}")

    plt.figure(figsize=(8, 5))
    plt.imshow(induction_scores.numpy(), cmap="viridis", aspect="auto")
    plt.colorbar(label="Induction score")
    plt.xlabel("Head")
    plt.ylabel("Layer")
    plt.title("Induction score per head (GPT-2 small)")
    plt.savefig("outputs/project1_induction_heads.png", dpi=150)
    print("Saved outputs/project1_induction_heads.png")

    return best_layer, best_head, induction_scores


# ---------------------------------------------------------------------------
# Project 2: Logit lens on a factual prompt
# ---------------------------------------------------------------------------
def project_2_logit_lens(model, prompt="The Eiffel Tower is located in the city of",
                          correct_answer=" Paris"):
    """
    At each layer, project the residual stream through the final layer norm
    and unembedding matrix, and track the rank of the correct token.
    Shows which layer the model "commits" to the answer.
    """
    tokens = model.to_tokens(prompt)
    correct_token = model.to_single_token(correct_answer)

    _, cache = model.run_with_cache(tokens)
    resid_stack, labels = cache.accumulated_resid(
        layer=-1, incl_mid=True, return_labels=True
    )
    # resid_stack: [n_components, batch, pos, d_model] — take last token position
    last_pos_resid = resid_stack[:, 0, -1, :]  # [n_components, d_model]

    ranks = []
    for i in range(last_pos_resid.shape[0]):
        normed = cache.apply_ln_to_stack(
            last_pos_resid[i].unsqueeze(0).unsqueeze(0), layer=-1
        )
        logits = normed @ model.W_U + model.b_U
        logits = logits.squeeze()
        rank = (logits > logits[correct_token]).sum().item()
        ranks.append(rank)

    print(f"\n[Project 2] Prompt: '{prompt}'")
    print(f"Correct answer token: '{correct_answer.strip()}'")
    print("Rank of correct token at each layer (0 = top prediction):")
    for label, rank in zip(labels, ranks):
        print(f"  {label}: rank {rank}")

    plt.figure(figsize=(8, 5))
    plt.plot(range(len(ranks)), ranks, marker="o")
    plt.xticks(range(len(labels)), labels, rotation=90)
    plt.ylabel("Rank of correct token (lower = more confident)")
    plt.title(f"Logit lens: '{prompt.strip()}' -> '{correct_answer.strip()}'")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig("outputs/project2_logit_lens.png", dpi=150)
    print("Saved outputs/project2_logit_lens.png")

    return ranks, labels


# ---------------------------------------------------------------------------
# Project 3: Ablate the induction head, measure the damage
# ---------------------------------------------------------------------------
def project_3_ablate_head(model, layer=None, head=None, seq_len=25, seed=0):
    """
    Zero out the induction head found in Project 1 and measure how much
    accuracy drops on predicting the repeated-sequence continuation.
    """
    if layer is None or head is None:
        layer, head, _ = project_1_induction_head(model, seq_len=seq_len, seed=seed)

    torch.manual_seed(seed)
    random_tokens = torch.randint(1000, 10000, (1, seq_len))
    repeated_tokens = torch.cat([random_tokens, random_tokens], dim=1)

    def get_accuracy(hooks=None):
        if hooks:
            logits = model.run_with_hooks(repeated_tokens, fwd_hooks=hooks)
        else:
            logits = model(repeated_tokens)
        preds = logits[0, seq_len - 1: -1].argmax(dim=-1)
        targets = repeated_tokens[0, seq_len:]
        return (preds == targets).float().mean().item()

    baseline_acc = get_accuracy()

    def zero_head_hook(value, hook):
        value[:, :, head, :] = 0.0
        return value

    hook_name = f"blocks.{layer}.attn.hook_z"
    ablated_acc = get_accuracy(hooks=[(hook_name, zero_head_hook)])

    print(f"\n[Project 3] Ablating L{layer}H{head}")
    print(f"  Baseline next-token accuracy on repeated sequence: {baseline_acc:.3f}")
    print(f"  Accuracy with head ablated:                        {ablated_acc:.3f}")
    print(f"  Drop: {baseline_acc - ablated_acc:.3f}")

    plt.figure(figsize=(4, 5))
    plt.bar(["Baseline", "Ablated"], [baseline_acc, ablated_acc],
            color=["steelblue", "indianred"])
    plt.ylabel("Next-token accuracy on repeated seq")
    plt.title(f"Effect of ablating L{layer}H{head}")
    plt.savefig("outputs/project3_ablation.png", dpi=150)
    print("Saved outputs/project3_ablation.png")

    return baseline_acc, ablated_acc


# ---------------------------------------------------------------------------
# Project 4: Activation patching (IOI-style)
# ---------------------------------------------------------------------------
def project_4_activation_patching(model):
    """
    Two prompts differing only in which name is mentioned first:
      clean: "When Mary and John went to the store, John gave a drink to"
      corrupted: "When John and Mary went to the store, John gave a drink to"
    Both have correct answer "Mary". Patch the corrupted run's residual
    stream, one layer at a time, with the clean run's activations at the
    final token position, and see which layer restores the correct answer.
    """
    clean_prompt = "When Mary and John went to the store, John gave a drink to"
    corrupted_prompt = "When John and Mary went to the store, John gave a drink to"
    correct_token = model.to_single_token(" Mary")
    incorrect_token = model.to_single_token(" John")

    clean_tokens = model.to_tokens(clean_prompt)
    corrupted_tokens = model.to_tokens(corrupted_prompt)

    def logit_diff(logits):
        final = logits[0, -1]
        return (final[correct_token] - final[incorrect_token]).item()

    clean_logits, clean_cache = model.run_with_cache(clean_tokens)
    corrupted_logits = model(corrupted_tokens)

    clean_diff = logit_diff(clean_logits)
    corrupted_diff = logit_diff(corrupted_logits)

    print(f"\n[Project 4] Clean logit diff (Mary - John): {clean_diff:.3f}")
    print(f"Corrupted logit diff (Mary - John):          {corrupted_diff:.3f}")

    n_layers = model.cfg.n_layers
    patched_diffs = []

    for layer in range(n_layers):
        def patch_hook(value, hook, layer=layer):
            value[:, -1, :] = clean_cache["resid_post", layer][:, -1, :]
            return value

        hook_name = f"blocks.{layer}.hook_resid_post"
        patched_logits = model.run_with_hooks(
            corrupted_tokens, fwd_hooks=[(hook_name, patch_hook)]
        )
        patched_diffs.append(logit_diff(patched_logits))

    print("Logit diff after patching clean activations into corrupted run, by layer:")
    for l, d in enumerate(patched_diffs):
        print(f"  Layer {l}: {d:.3f}")

    plt.figure(figsize=(8, 5))
    plt.plot(range(n_layers), patched_diffs, marker="o", label="Patched")
    plt.axhline(clean_diff, color="green", linestyle="--", label="Clean")
    plt.axhline(corrupted_diff, color="red", linestyle="--", label="Corrupted")
    plt.xlabel("Layer patched")
    plt.ylabel("Logit diff (Mary - John)")
    plt.title("Activation patching: which layer carries the name info?")
    plt.legend()
    plt.savefig("outputs/project4_patching.png", dpi=150)
    print("Saved outputs/project4_patching.png")

    return patched_diffs, clean_diff, corrupted_diff


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--project", default="all",
                         choices=["1", "2", "3", "4", "all"])
    args = parser.parse_args([])

    model = load_model()

    if args.project in ("1", "all"):
        project_1_induction_head(model)
    if args.project in ("2", "all"):
        project_2_logit_lens(model)
    if args.project in ("3", "all"):
        project_3_ablate_head(model)
    if args.project in ("4", "all"):
        project_4_activation_patching(model)


usage: colab_kernel_launcher.py [-h] [--project {1,2,3,4,all}]
colab_kernel_launcher.py: error: unrecognized arguments: -f /root/.local/share/jupyter/runtime/kernel-b2d2f76b-0243-4c61-8510-794883b20e94.json


SystemExit: 2

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
